<a href="https://colab.research.google.com/github/VOIDxIIO/Colab-with-gui/blob/main/colab-with-gui.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@title 🚀 Colab Desktop (Full Output Version)
#@markdown Shows every command and every error. Good for debugging.
#@markdown ---
#@markdown **Set your RustDesk password:**
RUSTDESK_PASSWORD = "colab1234" #@param {type:"string"}
#@markdown **Install Google Chrome?**
INSTALL_CHROME = True #@param {type:"boolean"}
#@markdown ---

import subprocess, os, time, IPython
from google.colab import output

def run(cmd):
    print(f"\n$ {cmd}")
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout.strip():
        print(p.stdout)
    if p.stderr.strip():
        print("STDERR:", p.stderr)
    print(f"-> exit code: {p.returncode}")
    return p.returncode

def bg(cmd, log, env=None):
    subprocess.Popen(cmd, shell=True,
        stdout=open(log, "w"), stderr=subprocess.STDOUT,
        env={**os.environ, **(env or {})},
        start_new_session=True)

print("="*60)
print("STEP 1/6: Installing base packages (~5-10 min)")
print("="*60)
run("apt-get update")
run("apt-get install -y dbus-x11 xvfb xfce4 xfce4-goodies wget curl")

print("\n" + "="*60)
print("STEP 2/6: Installing RustDesk")
print("="*60)
run("wget -q https://github.com/rustdesk/rustdesk/releases/download/1.5.0/rustdesk-1.5.0-x86_64.deb -O /content/rustdesk.deb")
run("ls -lh /content/rustdesk.deb")
run("apt-get install -y /content/rustdesk.deb")
run("which rustdesk")
run("touch /usr/lib/x86_64-linux-gnu/libcuda.so.1 && ldconfig")

if INSTALL_CHROME:
    print("\n" + "="*60)
    print("STEP 3/6: Installing Google Chrome")
    print("="*60)
    run("wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb -O /content/chrome.deb")
    run("ls -lh /content/chrome.deb")
    run("apt-get install -y /content/chrome.deb")
    run("sed -i 's|^Exec=/usr/bin/google-chrome-stable|Exec=/usr/bin/google-chrome-stable --no-sandbox --disable-dev-shm-usage --disable-gpu|' /usr/share/applications/google-chrome.desktop")
    run("update-desktop-database -q")

print("\n" + "="*60)
print("STEP 4/6: Starting virtual display + XFCE")
print("="*60)
run("pkill -9 Xvfb; pkill -9 rustdesk; pkill -9 xfce4; sleep 2")

bg("Xvfb :99 -screen 0 1280x720x24", "/content/xvfb.log")
time.sleep(3)
bg("dbus-daemon --system --fork", "/content/dbus.log")
time.sleep(2)

env = {"DISPLAY": ":99"}
bg("startxfce4", "/content/xfce.log", env)
time.sleep(10)
run("ps aux | grep -E 'Xvfb|xfce4' | grep -v grep")

print("\n" + "="*60)
print("STEP 5/6: Starting RustDesk")
print("="*60)
bg("rustdesk --server", "/content/rd_server.log", env)
time.sleep(8)
run(f"rustdesk --password {RUSTDESK_PASSWORD}")
bg("rustdesk --headless", "/content/rd_headless.log", env)
time.sleep(3)
run("ps aux | grep rustdesk | grep -v grep")

print("\n" + "="*60)
print("STEP 6/6: Waiting for RustDesk to register")
print("="*60)
rd_id = ""
for i in range(30):
    time.sleep(2)
    r = subprocess.run("rustdesk --get-id", shell=True, capture_output=True, text=True)
    rd_id = r.stdout.strip()
    if rd_id:
        print(f"✅ Registered after {i*2}s")
        break
    print(f"   ... waiting ({i*2}s)")

print("\n--- rustdesk server log ---")
run("cat /content/rd_server.log")
print("\n--- xfce log (last 20 lines) ---")
run("tail -20 /content/xfce.log")

# keep-alive
display(IPython.display.Javascript('''
setInterval(function(){
  var b = document.querySelector("colab-connect-button");
  if (b) b.click();
  var ok = document.getElementById("ok");
  if (ok) ok.click();
}, 60000);
'''))

print("\n" + "="*55)
print("✅  DONE")
print("="*55)
print(f"  RustDesk ID :  {rd_id if rd_id else '❌ FAILED'}")
print(f"  Password    :  {RUSTDESK_PASSWORD}")
print("="*55)